# Named Entity Recognition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/nlp-learning-journey/blob/main/examples/spaCy-Linguistic/04-spaCy-Linguistic-Named-Entity-Recognition.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/vuhung16au/nlp-learning-journey/blob/main/examples/spaCy-Linguistic/04-spaCy-Linguistic-Named-Entity-Recognition.ipynb)
[![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/vuhung16au/nlp-learning-journey/blob/main/examples/spaCy-Linguistic/04-spaCy-Linguistic-Named-Entity-Recognition.ipynb)

**Author:** spaCy Tutorial Series  
**Created:** 2024-01-01  
**Last Modified:** 2024-01-01  
**Tags:** spacy, nlp, python, tutorial, intermediate  
**Duration:** 45-60 minutes  
**Level:** Intermediate  

## Learning Objectives
By the end of this notebook, you will be able to:
- Understand what named entities are
- Extract entities from text
- Work with different entity types
- Visualize named entities
- Apply NER to real-world problems

## Table of Contents
1. [Named Entity Recognition 101](#named-entity-recognition-101)
2. [Working with Entities](#working-with-entities)
3. [Visualizing Named Entities](#visualizing-named-entities)
4. [Practice Exercises](#practice-exercises)
5. [Summary and Key Takeaways](#summary-and-key-takeaways)


In [ ]:
# Environment Detection and Setup
import sys
import subprocess
import os

# Detect the runtime environment
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

print(f"Environment detected:")
print(f"  - Local: {IS_LOCAL}")
print(f"  - Google Colab: {IS_COLAB}")
print(f"  - Kaggle: {IS_KAGGLE}")

In [ ]:
# Platform-specific spaCy installation and setup
if IS_COLAB:
    print("\nSetting up Google Colab environment...")
    !pip install -q spacy
    !python -m spacy download en_core_web_sm
elif IS_KAGGLE:
    print("\nSetting up Kaggle environment...")
    # Kaggle usually has spaCy pre-installed
    !python -m spacy download en_core_web_sm
else:
    print("\nSetting up local environment...")
    # For local environment, packages should be installed via requirements.txt
    # Verify spaCy model is available
    try:
        import spacy
        nlp = spacy.load("en_core_web_sm")
        print("✓ spaCy and en_core_web_sm model are available")
    except:
        print("⚠ Please run: python -m spacy download en_core_web_sm")

In [ ]:
import spacy
from spacy import displacy

# Load the English language model
nlp = spacy.load("en_core_web_sm")


## 1. Named Entity Recognition 101

### What are Named Entities?

Named entities are real-world objects that can be denoted with a proper name, such as:
- **PERSON**: People (Barack Obama, Marie Curie)
- **ORG**: Organizations (Apple Inc., United Nations)
- **GPE**: Geopolitical entities (United States, France)
- **LOC**: Locations (Mount Everest, Pacific Ocean)
- **DATE**: Dates (January 1st, 2023)
- **MONEY**: Monetary values ($100, €50)


### Built-in Entity Types

spaCy recognizes many entity types. Here are the most common ones:


In [ ]:
# Explore different entity types
text = "Apple Inc. was founded by Steve Jobs in California on April 1, 1976. The company is worth $2 trillion."
doc = nlp(text)

print("Entity Types and Examples:")
print("=" * 35)
for ent in doc.ents:
    print(f"'{ent.text}' -> {ent.label_} (confidence: {ent.label})")

# Group entities by type
entity_types = {}
for ent in doc.ents:
    if ent.label_ not in entity_types:
        entity_types[ent.label_] = []
    entity_types[ent.label_].append(ent.text)

print(f"\nEntities grouped by type:")
for label, entities in entity_types.items():
    print(f"{label}: {', '.join(entities)}")


## 2. Working with Entities

### Extracting Entities from Text

Let's explore how to work with entities in more detail:


In [ ]:
# Working with entity properties
text = "Barack Obama was the 44th President of the United States from 2009 to 2017."
doc = nlp(text)

print("Entity Properties:")
print("=" * 25)
for ent in doc.ents:
    print(f"Text: '{ent.text}'")
    print(f"  - Label: {ent.label_}")
    print(f"  - Start: {ent.start_char}, End: {ent.end_char}")
    print(f"  - Span: {ent.start} - {ent.end}")
    print(f"  - Confidence: {ent.label}")
    print()


### Filtering by Entity Type

You can filter entities by their type for specific analysis:


In [ ]:
# Filter entities by type
text = "Apple Inc. was founded by Steve Jobs in California. The company is worth $2 trillion and employs 150,000 people."
doc = nlp(text)

print("Filtered Entities:")
print("=" * 20)

# Extract only people
people = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
print(f"People: {people}")

# Extract only organizations
organizations = [ent.text for ent in doc.ents if ent.label_ == "ORG"]
print(f"Organizations: {organizations}")

# Extract only locations
locations = [ent.text for ent in doc.ents if ent.label_ == "GPE"]
print(f"Locations: {locations}")

# Extract only monetary values
money = [ent.text for ent in doc.ents if ent.label_ == "MONEY"]
print(f"Money: {money}")

# Extract only cardinal numbers
numbers = [ent.text for ent in doc.ents if ent.label_ == "CARDINAL"]
print(f"Numbers: {numbers}")


### Entity Relationships

You can also analyze relationships between entities and other parts of the text:


In [ ]:
# Analyze entity relationships
text = "Tim Cook is the CEO of Apple Inc. in Cupertino, California."
doc = nlp(text)

print("Entity Relationships:")
print("=" * 25)

for ent in doc.ents:
    print(f"Entity: '{ent.text}' ({ent.label_})")
    
    # Find the root of the entity
    root = ent.root
    print(f"  Root: '{root.text}' (POS: {root.pos_})")
    
    # Find what the entity is connected to
    for token in ent:
        if token.head != token:
            print(f"  Connected to: '{token.head.text}' via '{token.dep_}'")
    
    print()


## 3. Visualizing Named Entities

### Using displacy for NER Visualization

spaCy's displacy module provides excellent visualization tools for named entities:


In [ ]:
# Basic NER visualization
text = "Apple Inc. was founded by Steve Jobs in California on April 1, 1976."
doc = nlp(text)

# Create NER visualization
displacy.render(doc, style="ent", jupyter=True)


### Styling and Customization

You can customize the visualization with different colors and styles:


In [ ]:
# Custom styling for NER visualization
text = "Microsoft Corporation was founded by Bill Gates in Seattle, Washington."
doc = nlp(text)

# Custom colors for different entity types
colors = {
    "PERSON": "#ff9999",
    "ORG": "#99ccff", 
    "GPE": "#99ff99",
    "DATE": "#ffcc99"
}

# Create custom options
options = {
    "ents": ["PERSON", "ORG", "GPE", "DATE"],
    "colors": colors
}

# Render with custom styling
displacy.render(doc, style="ent", jupyter=True, options=options)


## 4. Practice Exercises

### Exercise 1: Entity Extraction Function
Create a function that extracts entities of a specific type from text.


In [ ]:
# Your solution here
def extract_entities_by_type(text, entity_type, nlp_model):
    """
    Extract entities of a specific type from text
    """
    doc = nlp_model(text)
    entities = [ent.text for ent in doc.ents if ent.label_ == entity_type]
    return entities

# Test the function
test_text = "Apple Inc. was founded by Steve Jobs in California. Microsoft was founded by Bill Gates in Seattle."
people = extract_entities_by_type(test_text, "PERSON", nlp)
organizations = extract_entities_by_type(test_text, "ORG", nlp)
locations = extract_entities_by_type(test_text, "GPE", nlp)

print("Extracted Entities:")
print("=" * 20)
print(f"People: {people}")
print(f"Organizations: {organizations}")
print(f"Locations: {locations}")


### Exercise 2: Entity Statistics
Create a function that provides statistics about entities in text.


In [ ]:
# Your solution here
def get_entity_statistics(text, nlp_model):
    """
    Get statistics about entities in text
    """
    doc = nlp_model(text)
    
    # Count entities by type
    entity_counts = {}
    for ent in doc.ents:
        if ent.label_ not in entity_counts:
            entity_counts[ent.label_] = 0
        entity_counts[ent.label_] += 1
    
    # Get unique entities
    unique_entities = set(ent.text for ent in doc.ents)
    
    # Get most common entity type
    most_common_type = max(entity_counts.items(), key=lambda x: x[1]) if entity_counts else ("None", 0)
    
    return {
        "total_entities": len(doc.ents),
        "unique_entities": len(unique_entities),
        "entity_counts": entity_counts,
        "most_common_type": most_common_type
    }

# Test the function
test_text = "Apple Inc. was founded by Steve Jobs in California. Microsoft was founded by Bill Gates in Seattle. Both companies are technology giants."
stats = get_entity_statistics(test_text, nlp)

print("Entity Statistics:")
print("=" * 20)
print(f"Total entities: {stats['total_entities']}")
print(f"Unique entities: {stats['unique_entities']}")
print(f"Entity counts: {stats['entity_counts']}")
print(f"Most common type: {stats['most_common_type'][0]} ({stats['most_common_type'][1]} occurrences)")


### Exercise 3: Custom Entity Recognition
Create a simple custom entity recognizer using spaCy's EntityRuler.


In [ ]:
# Your solution here
from spacy.pipeline import EntityRuler

def create_custom_ner():
    """
    Create a custom NER model with EntityRuler
    """
    # Create a blank English model
    nlp_custom = spacy.blank("en")
    
    # Add the EntityRuler to the pipeline
    ruler = nlp_custom.add_pipe("entity_ruler")
    
    # Define custom patterns
    patterns = [
        {"label": "PROGRAMMING_LANGUAGE", "pattern": [{"LOWER": "python"}]},
        {"label": "PROGRAMMING_LANGUAGE", "pattern": [{"LOWER": "javascript"}]},
        {"label": "PROGRAMMING_LANGUAGE", "pattern": [{"LOWER": "java"}]},
        {"label": "TECH_COMPANY", "pattern": [{"LOWER": "google"}]},
        {"label": "TECH_COMPANY", "pattern": [{"LOWER": "facebook"}]},
        {"label": "TECH_COMPANY", "pattern": [{"LOWER": "amazon"}]},
    ]
    
    # Add patterns to the ruler
    ruler.add_patterns(patterns)
    
    return nlp_custom

# Test the custom NER
nlp_custom = create_custom_ner()
test_text = "I love programming in Python and JavaScript. Google and Facebook are tech giants."
doc = nlp_custom(test_text)

print("Custom Entity Recognition:")
print("=" * 30)
for ent in doc.ents:
    print(f"'{ent.text}' -> {ent.label_}")


## 5. Summary and Key Takeaways

### What We Learned

1. **Named Entity Recognition**: Understanding what entities are and how to extract them
2. **Entity Types**: Working with different built-in entity types (PERSON, ORG, GPE, etc.)
3. **Entity Properties**: Accessing entity text, labels, positions, and confidence scores
4. **Entity Filtering**: Extracting entities of specific types
5. **Entity Visualization**: Using displacy for NER visualization
6. **Custom NER**: Creating custom entity recognition with EntityRuler

### Key Concepts

- **Named Entities**: Real-world objects with proper names
- **Entity Types**: Categories like PERSON, ORG, GPE, DATE, MONEY
- **Entity Properties**: Text, label, position, confidence
- **Entity Filtering**: Selecting entities by type
- **Entity Visualization**: Interactive NER displays
- **Custom Patterns**: Rule-based entity recognition

### Practical Applications

- **Information Extraction**: Pulling structured data from unstructured text
- **Text Analysis**: Understanding content through entity analysis
- **Data Mining**: Extracting insights from large text corpora
- **Content Classification**: Categorizing text based on entities
- **Search and Retrieval**: Improving search with entity recognition

### Best Practices

1. **Choose Appropriate Models**: Use models trained on relevant domains
2. **Validate Results**: Check entity recognition accuracy for your use case
3. **Handle Ambiguity**: Be aware that some entities may be misclassified
4. **Custom Patterns**: Use EntityRuler for domain-specific entities
5. **Performance**: Consider processing speed for large datasets

### Next Steps

- Explore entity linking (connecting entities to knowledge bases)
- Learn about custom model training for NER
- Integrate NER with other NLP tasks
- Build entity-based applications

### Common Pitfalls

1. **Model Limitations**: Not all entities are recognized correctly
2. **Domain Specificity**: Models may not work well for specialized domains
3. **Ambiguous Entities**: Some text may be misclassified
4. **Performance**: NER can be computationally expensive
5. **Language Support**: NER quality varies by language


## Cross-lingual Example: Vietnamese/English NER

According to the repository's language focus, we prioritize Vietnamese/English examples for cross-lingual NLP tasks.

**Note**: spaCy's `en_core_web_sm` model is trained on English text. For Vietnamese NER, you would need a Vietnamese language model or a multilingual model. This example demonstrates the concept with English NER, and shows how Vietnamese text would be structured.

In [ ]:
# English Named Entity Recognition Example
english_text = "My name is John and I live in Vietnam. I work at Google."
doc_en = nlp(english_text)

print("English NER Results:")
print("=" * 50)
for ent in doc_en.ents:
    print(f"Text: '{ent.text}'")
    print(f"  - Label: {ent.label_} ({spacy.explain(ent.label_)})")
    print(f"  - Start: {ent.start_char}, End: {ent.end_char}")
    print()

# Vietnamese text example (would require Vietnamese language model)
vietnamese_text = "Tên tôi là John và tôi sống ở Việt Nam. Tôi làm việc tại Google."
print("\nVietnamese Text (Translation):")
print("=" * 50)
print(f"Vietnamese: {vietnamese_text}")
print(f"English: {english_text}")
print("\n⚠ Note: Vietnamese NER requires a Vietnamese language model.")
print("For production use, consider:")
print("  - spaCy Vietnamese models (if available)")
print("  - Multilingual models like XLM-RoBERTa")
print("  - Vietnamese-specific NLP libraries")

In [ ]:
# Basic NER example
text = "Apple Inc. was founded by Steve Jobs in California on April 1, 1976."
doc = nlp(text)

print("Named Entity Recognition:")
print("=" * 30)
for ent in doc.ents:
    print(f"'{ent.text}' -> {ent.label_} (confidence: {ent.label})")
